<a href="https://colab.research.google.com/github/Yusufaltnbs/bert-turkish-sentiment-analysis/blob/main/bert-turkish-sentiment-analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import pandas as pd


dataset = load_dataset("winvoker/turkish-sentiment-analysis-dataset")


df = pd.DataFrame(dataset['train'])


df = df[df['label'] != 'Notr']

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/76.1M [00:00<?, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/440679 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/48965 [00:00<?, ? examples/s]

In [2]:
df = df.drop(columns=['dataset'])
label_map = {'Positive': 1, 'Negative': 0}
df['label'] = df['label'].map(label_map)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print("Toplam veri:", len(df))
print("Eğitim verisi:", len(train_df))
print("Test verisi:", len(test_df))

Toplam veri: 286854
Eğitim verisi: 229483
Test verisi: 57371


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset
import torch

model_name = "dbmdz/bert-base-turkish-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)


def tokenize_function(example):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=128)


train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)


train_dataset = train_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/229483 [00:00<?, ? examples/s]

Map:   0%|          | 0/57371 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dbmdz/bert-base-turkish-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
from transformers import TrainingArguments, Trainer, EvalPrediction
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred: EvalPrediction):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

/tmp/ipython-input-2442251705.py:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [8]:
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.133300,0.137512,0.958516,0.975117
2,0.069800,0.150492,0.961409,0.976695


TrainOutput(global_step=28686, training_loss=0.12417987640421152, metrics={'train_runtime': 10228.6018, 'train_samples_per_second': 44.871, 'train_steps_per_second': 2.804, 'total_flos': 3.018975715858944e+16, 'train_loss': 0.12417987640421152, 'epoch': 2.0})

In [12]:

results = trainer.evaluate()
print(results)


{'eval_loss': 0.15049244463443756, 'eval_accuracy': 0.9614090742709731, 'eval_f1': 0.9766947368421053, 'eval_runtime': 393.0387, 'eval_samples_per_second': 145.968, 'eval_steps_per_second': 9.124, 'epoch': 2.0}


In [11]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from huggingface_hub import login

login()

save_path = "./my_trained_model"
trainer.save_model(save_path)
# yeni yöntem
tokenizer.save_pretrained("./my_trained_model")

model.push_to_hub("bert-turkish-sentiment-analysis")
tokenizer.push_to_hub("bert-turkish-sentiment-analysis")

print("Model ve tokenizer Hugging Face'e yüklendi!")
print("Repo linki: https://huggingface.co/<username>/bert-turkish-sentiment-analysis")



Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...9rjp137/model.safetensors:   0%|          |  555kB /  442MB            

README.md: 0.00B [00:00, ?B/s]

✅ Model ve tokenizer Hugging Face'e yüklendi!
Repo linki: https://huggingface.co/<username>/bert-turkish-sentiment-analysis
